In [1]:
!pip install "openai>=1.0.0"


In [1]:
!pip install "openai>=1.0.0" pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.9 MB/s eta 0:00:00


In [ ]:
import os
from openai import OpenAI
import pandas as pd
import time
import random

# Initialize the OpenAI client
# Make sure OPENAI_API_KEY is set in your environment variables
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))


In [3]:
prompts = [
    "Write a {sentiment} review about the website's {aspect}:",
    "How would you describe the {aspect} of the website in a {sentiment} tone?",
    "Give a {sentiment} opinion on the {aspect} of the website:"
]

aspects = ["design", "navigation", "speed", "functionality", "usability"]
sentiments = ["positive", "negative", "neutral"]
features = ["homepage layout", "menu structure", "loading time", "search function", "overall ease of use"]
num_reviews_per_combination = 10  # Adjust this to control the number of reviews generated per combination

In [6]:
def generate_reviews(prompts, aspects, sentiments, features, num_reviews_per_combination):
    reviews = []
    for aspect in aspects:
        for sentiment in sentiments:
            for feature in features:
                for _ in range(num_reviews_per_combination):
                    prompt = random.choice(prompts).format(sentiment=sentiment, aspect=aspect)
                    try:
                        response = client.chat.completions.create(
                            model="gpt-3.5-turbo",  # Updated to modern chat model
                            messages=[
                                {"role": "system", "content": "You are a helpful assistant."},
                                {"role": "user", "content": prompt}
                            ],
                            max_tokens=100,
                            n=1,
                            temperature=0.7
                        )
                        review = response.choices[0].message.content.strip()
                        reviews.append((review, sentiment, aspect, feature))
                    except Exception as e:
                        print(f"Error generating review: {e}")
                    time.sleep(1)  # To avoid rate limits
    return reviews

# Generate a small dataset for testing
total_reviews = 10  # Reduced for quick testing
reviews = []

# Calculate the number of iterations needed
iterations = max(1, total_reviews // (len(aspects) * len(sentiments) * len(features) * num_reviews_per_combination))
for _ in range(iterations):
    batch_reviews = generate_reviews(prompts, aspects, sentiments, features, num_reviews_per_combination)
    reviews.extend(batch_reviews)
    break # Only run one iteration for testing

# Create a DataFrame to store the synthetic reviews
df = pd.DataFrame(reviews, columns=["review", "sentiment", "aspect", "feature"])

# Display the first few rows
print(df.head())
print(f"Total reviews generated: {df.shape[0]}")

# Save to CSV
df.to_csv("synthetic_usability_reviews.csv", index=False)


InvalidRequestError: The model `davinci` has been deprecated, learn more here: https://platform.openai.com/docs/deprecations